# Tokyo 2020 Olympics: PySpark ETL and Analysis

**What this notebook does**
1. **Extract:** read 5 raw CSV files (Athletes, Coaches, EntriesGender, Medals, Teams) with PySpark.
2. **Transform:** check data quality, fix text encoding, clean names and types, remove duplicates.
3. **Load:** save the data in two layers as Parquet: `data/cleansed` (cleaned tables) and `data/curated` (tables ready for analysis).
4. **Analyze:** answer questions with Spark SQL and draw charts, saved in `images/`.

> Run it in Google Colab: **Runtime, then Run all**. It takes about 2 to 3 minutes.

> Background: this project was first built on Azure (Data Factory, Databricks, Synapse) by following a tutorial. Azure access has ended, so this notebook is a local PySpark rebuild of the same ETL and analysis.

## 0. Setup

In [ ]:
import importlib.util, subprocess, sys

# Install PySpark only if it is not already installed (Colab usually needs this step).
if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
print("PySpark ready")

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/SaeeR18/Olympic-Data-Analysis-Project.git"
REPO_DIR = "Olympic-Data-Analysis-Project"

# Download the repo (it holds the raw CSV files). Skip if we are already inside it.
if os.path.basename(os.getcwd()) != REPO_DIR and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

# Raw files are in data/raw if you moved them there, otherwise in the repo root.
RAW = "data/raw" if os.path.isdir("data/raw") else "."
for folder in ["data/cleansed", "data/curated", "images"]:
    os.makedirs(folder, exist_ok=True)

print("Working folder:", os.getcwd())
print("Reading raw CSVs from:", RAW)

In [ ]:
from pyspark.sql import SparkSession, functions as F

# A Spark session is the entry point to Spark. "local[*]" means: use all CPU cores on this machine.
spark = (SparkSession.builder
         .appName("OlympicETL")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")   # small data, so fewer partitions
         .getOrCreate())
print("Spark version:", spark.version)

## 1. Extract: read the raw CSV files

Three files (Athletes, Coaches, Teams) are saved in **ISO-8859-1** encoding, not UTF-8. If we read them as UTF-8, names like *Côte d'Ivoire* turn into broken characters. We found this during the quality check, so each file is read with the right encoding.

In [ ]:
def read_csv(name, encoding="UTF-8"):
    return (spark.read
            .option("header", True)        # first row holds column names
            .option("inferSchema", True)   # let Spark guess number columns
            .option("encoding", encoding)
            .csv(f"{RAW}/{name}.csv"))

athletes_raw = read_csv("Athletes", "ISO-8859-1")
coaches_raw  = read_csv("Coaches",  "ISO-8859-1")
teams_raw    = read_csv("Teams",    "ISO-8859-1")
entries_raw  = read_csv("EntriesGender")
medals_raw   = read_csv("Medals")

raw_tables = {"athletes": athletes_raw, "coaches": coaches_raw, "teams": teams_raw,
              "entries_gender": entries_raw, "medals": medals_raw}
for name, df in raw_tables.items():
    print(f"{name:15s} rows={df.count():6d}  columns={df.columns}")

## 2. Data quality check (before cleaning)

In [ ]:
def null_counts(df):
    # For each column, count how many values are empty (null).
    return df.select([F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in df.columns])

for name, df in raw_tables.items():
    print(name)
    null_counts(df).show(truncate=False)

In [ ]:
# Duplicate rows (exact copies) in each table
for name, df in raw_tables.items():
    print(f"{name:15s} duplicate rows: {df.count() - df.dropDuplicates().count()}")

# Medals file: does gold + silver + bronze equal total?
bad_totals = medals_raw.filter(F.col("Gold") + F.col("Silver") + F.col("Bronze") != F.col("Total")).count()
print("Medals rows where gold+silver+bronze != total:", bad_totals)

# EntriesGender file: does female + male equal total?
bad_entries = entries_raw.filter(F.col("Female") + F.col("Male") != F.col("Total")).count()
print("EntriesGender rows where female+male != total:", bad_entries)

In [ ]:
# A join problem to watch: do all countries in Medals also appear in Athletes?
medal_countries = {r[0] for r in medals_raw.select("TeamCountry").distinct().collect()}
athlete_countries = {r[0] for r in athletes_raw.select("Country").distinct().collect()}
print("Countries in Medals but not in Athletes:", medal_countries - athlete_countries)

## 3. Transform: clean the data

Cleaning steps:
- Trim extra spaces in text columns.
- Rename columns to simple `snake_case` names.
- Fix broken characters in the Medals country names (this file was saved with the *Côte d'Ivoire* text already damaged).
- Fill empty `event` values in Coaches with `Not specified`.
- Remove exact duplicate rows.
- Add `female_pct` (share of female entries) to the gender table.

In [ ]:
def trim_strings(df):
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))
    return df

athletes = (trim_strings(athletes_raw)
            .withColumnRenamed("PersonName", "person_name")
            .withColumnRenamed("Country", "country")
            .withColumnRenamed("Discipline", "discipline")
            .dropDuplicates())   # note: could also remove two different people with identical rows (very rare)

coaches = (trim_strings(coaches_raw)
           .withColumnRenamed("Name", "coach_name")
           .withColumnRenamed("Country", "country")
           .withColumnRenamed("Discipline", "discipline")
           .withColumnRenamed("Event", "event")
           .fillna({"event": "Not specified"})
           .dropDuplicates())

teams = (trim_strings(teams_raw)
         .withColumnRenamed("TeamName", "team_name")
         .withColumnRenamed("Discipline", "discipline")
         .withColumnRenamed("Country", "country")
         .withColumnRenamed("Event", "event")
         .dropDuplicates())

entries_gender = (trim_strings(entries_raw)
                  .withColumnRenamed("Discipline", "discipline")
                  .withColumnRenamed("Female", "female")
                  .withColumnRenamed("Male", "male")
                  .withColumnRenamed("Total", "total")
                  .withColumn("female_pct", F.round(F.col("female") / F.col("total") * 100, 1))
                  .dropDuplicates())

medals = (trim_strings(medals_raw)
          .withColumnRenamed("Rank", "rank")
          .withColumnRenamed("TeamCountry", "country")
          .withColumnRenamed("Gold", "gold")
          .withColumnRenamed("Silver", "silver")
          .withColumnRenamed("Bronze", "bronze")
          .withColumnRenamed("Total", "total")
          .withColumnRenamed("Rank by Total", "rank_by_total")
          # fix damaged characters: "C\u00c3\u00b4te d'Ivoire" -> "C\u00f4te d'Ivoire"
          .withColumn("country", F.regexp_replace("country", "C\u00c3\u00b4te", "C\u00f4te"))
          .dropDuplicates())

clean_tables = {"athletes": athletes, "coaches": coaches, "teams": teams,
                "entries_gender": entries_gender, "medals": medals}

print(f"{'table':15s} {'rows before':>12s} {'rows after':>11s}")
for name in clean_tables:
    print(f"{name:15s} {raw_tables[name].count():12d} {clean_tables[name].count():11d}")

In [ ]:
# Re-check the join problem after cleaning
medal_countries = {r[0] for r in medals.select("country").distinct().collect()}
athlete_countries = {r[0] for r in athletes.select("country").distinct().collect()}
print("Countries in Medals but not in Athletes:", medal_countries - athlete_countries)

## 4. Load: save the cleansed layer (Parquet)

Parquet is a compressed column format that Spark reads quickly and that keeps column types.

In [ ]:
for name, df in clean_tables.items():
    df.write.mode("overwrite").parquet(f"data/cleansed/{name}")
    print("saved data/cleansed/" + name)

## 5. Curated layer: tables built for analysis

`country_summary` joins medals with the number of athletes, coaches and teams per country.

In [ ]:
athletes_by_country = athletes.groupBy("country").agg(F.count("*").alias("athletes"))
coaches_by_country  = coaches.groupBy("country").agg(F.count("*").alias("coaches"))
teams_by_country    = teams.groupBy("country").agg(F.count("*").alias("teams"))

country_summary = (medals
    .join(athletes_by_country, "country", "left")
    .join(coaches_by_country,  "country", "left")
    .join(teams_by_country,    "country", "left")
    .fillna(0, subset=["athletes", "coaches", "teams"])
    .withColumn("medals_per_100_athletes",
                F.when(F.col("athletes") > 0,
                       F.round(F.col("total") / F.col("athletes") * 100, 1))))

athletes_by_discipline = athletes.groupBy("discipline").agg(F.count("*").alias("athletes"))

country_summary.write.mode("overwrite").parquet("data/curated/country_summary")
entries_gender.write.mode("overwrite").parquet("data/curated/gender_by_discipline")
athletes_by_discipline.write.mode("overwrite").parquet("data/curated/athletes_by_discipline")

print("curated rows:", country_summary.count(), entries_gender.count(), athletes_by_discipline.count())
country_summary.orderBy("rank").show(5, truncate=False)

## 6. Analysis with Spark SQL

We read back from the Parquet files, register them as SQL views, and ask questions in SQL.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

spark.read.parquet("data/curated/country_summary").createOrReplaceTempView("country_summary")
spark.read.parquet("data/curated/gender_by_discipline").createOrReplaceTempView("gender_by_discipline")
spark.read.parquet("data/curated/athletes_by_discipline").createOrReplaceTempView("athletes_by_discipline")

def save_and_show(fig, filename):
    fig.tight_layout()
    fig.savefig(f"images/{filename}", dpi=150)
    plt.show()

### Q1. Which 10 countries won the most gold medals?

In [ ]:
q1 = spark.sql('''
    SELECT country, gold, silver, bronze, total
    FROM country_summary
    ORDER BY gold DESC, silver DESC
    LIMIT 10
''').toPandas()
display(q1) if "display" in globals() else print(q1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(q1["country"][::-1], q1["gold"][::-1], color="#d4a017")
ax.set_xlabel("Gold medals"); ax.set_title("Top 10 countries by gold medals")
save_and_show(fig, "01_top10_gold.png")

### Q2. How are gold, silver and bronze split for the top 10 countries by total medals?

In [ ]:
q2 = spark.sql('''
    SELECT country, gold, silver, bronze, total
    FROM country_summary
    ORDER BY total DESC
    LIMIT 10
''').toPandas()

fig, ax = plt.subplots(figsize=(9, 5))
y = q2["country"][::-1]
ax.barh(y, q2["gold"][::-1], color="#d4a017", label="Gold")
ax.barh(y, q2["silver"][::-1], left=q2["gold"][::-1], color="#9e9e9e", label="Silver")
ax.barh(y, q2["bronze"][::-1], left=(q2["gold"] + q2["silver"])[::-1], color="#a0522d", label="Bronze")
ax.set_xlabel("Medals"); ax.set_title("Top 10 countries by total medals"); ax.legend()
save_and_show(fig, "02_top10_total_stacked.png")

### Q3. Female and male entries by discipline (15 biggest disciplines)

In [ ]:
q3 = spark.sql('''
    SELECT discipline, female, male, total, female_pct
    FROM gender_by_discipline
    ORDER BY total DESC
    LIMIT 15
''').toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
pos = range(len(q3))
ax.bar([p - 0.2 for p in pos], q3["female"], width=0.4, label="Female", color="#c2185b")
ax.bar([p + 0.2 for p in pos], q3["male"], width=0.4, label="Male", color="#1565c0")
ax.set_xticks(list(pos)); ax.set_xticklabels(q3["discipline"], rotation=60, ha="right")
ax.set_ylabel("Entries"); ax.set_title("Female vs male entries, 15 biggest disciplines"); ax.legend()
save_and_show(fig, "03_gender_by_discipline.png")

totals = spark.sql("SELECT SUM(female) AS female, SUM(male) AS male, SUM(total) AS total FROM gender_by_discipline").toPandas()
print(totals)

### Q4. Which disciplines are most female-dominated and most male-dominated? (disciplines with at least 50 entries)

In [ ]:
q4 = spark.sql('''
    SELECT discipline, female, male, total, female_pct
    FROM gender_by_discipline
    WHERE total >= 50
    ORDER BY female_pct DESC
''').toPandas()
print("Highest female share:"); print(q4.head(5).to_string(index=False))
print("\nLowest female share:"); print(q4.tail(5).to_string(index=False))

### Q5. Which countries sent the most athletes, and which won the most medals per 100 athletes? (countries with at least 50 athletes)

In [ ]:
q5a = spark.sql('''
    SELECT country, athletes, total AS medals
    FROM country_summary
    ORDER BY athletes DESC
    LIMIT 10
''').toPandas()

q5b = spark.sql('''
    SELECT country, athletes, total AS medals, medals_per_100_athletes
    FROM country_summary
    WHERE athletes >= 50
    ORDER BY medals_per_100_athletes DESC
    LIMIT 10
''').toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(q5a["country"][::-1], q5a["athletes"][::-1], color="#2e7d32")
axes[0].set_title("Top 10 countries by number of athletes"); axes[0].set_xlabel("Athletes")
axes[1].barh(q5b["country"][::-1], q5b["medals_per_100_athletes"][::-1], color="#6a1b9a")
axes[1].set_title("Medals per 100 athletes (countries with 50+ athletes)"); axes[1].set_xlabel("Medals per 100 athletes")
save_and_show(fig, "04_athletes_and_efficiency.png")
print(q5b.to_string(index=False))

### Q6. Does sending more athletes mean winning more medals?

In [ ]:
q6 = spark.sql("SELECT country, athletes, total AS medals FROM country_summary WHERE athletes > 0").toPandas()
corr = q6["athletes"].corr(q6["medals"])
print("Correlation between athletes and medals:", round(corr, 2))

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(q6["athletes"], q6["medals"], alpha=0.7, color="#00838f")
for _, row in q6.sort_values("medals", ascending=False).head(6).iterrows():
    ax.annotate(row["country"], (row["athletes"], row["medals"]), fontsize=8)
ax.set_xlabel("Athletes"); ax.set_ylabel("Total medals"); ax.set_title("Athletes vs medals per country")
save_and_show(fig, "05_athletes_vs_medals.png")

## 7. Done

Files created by this notebook:
- `data/cleansed/` : 5 cleaned tables (Parquet)
- `data/curated/` : 3 analysis tables (Parquet)
- `images/` : 5 charts

Download this notebook **with its outputs** and push it, with the new folders, to GitHub.